In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
df = pd.read_csv('dataset_class.csv')

## עמודה N: vehicle_type
עמודה קטגוריאלית המציגה 6 סוגי רכבים המופיעים בדאטה.סוג הרכב ידוע ביום החתימה על ההסכם ולכן אין דליפת מידע. 

נבדוק את מספרי הנהגים בכל רכב, ערכים חסרים, ואת השפעת העמודה על משתנה המטרה

In [2]:
# בדיקת ערכים חסרים ורשימת סוגי הרכב עם מספר הנהגים בכל אחד

print("Missing:", df["vehicle_type"].isna().sum())
df["vehicle_type"].value_counts()

Missing: 0


vehicle_type
sedan         13916
small         12227
suv           10960
motorcycle     3072
pickup         2985
sports         1840
Name: count, dtype: int64

In [3]:
# השוואת עלות התביעות בין סוגי הרכב לפי שלושה מדדים
df.groupby("vehicle_type")["annual_claim_cost"].agg(
    avg_cost="mean",
    share_with_claim=lambda s: (s > 0).mean(),
    share_expensive=lambda s: (s > 10000).mean(),
    drivers="count",
).round(3).sort_values("avg_cost")

,avg_cost,share_with_claim,share_expensive,drivers
vehicle_type,,,,
small,1175.473,0.319,0.015,12227
sedan,2549.068,0.341,0.075,13916
motorcycle,2876.179,0.485,0.079,3072
suv,4051.931,0.338,0.143,10960
pickup,6483.999,0.435,0.214,2985
sports,8066.855,0.362,0.261,1840


### מסקנה לגבי vehicle_type
העמודה נקייה, עם פורמט אחיד וללא ערכים חסרים. יש הבדל גדול בעלות בין סוגי הרכב, פי 6.9 בין רכב קטן לרכב ספורט, העמודה תישאר.
סוגי הרכב שונים גם בתדירות התביעות וגם בחומרה. לאופנועים יש הכי הרבה תביעות (48.5%) אך מעט נהגים יקרים (7.9%). לרכבי ספורט יש אחוז תביעות רגיל (36%) אך הכי הרבה נהגים יקרים (26.1%).
לטנדרים יש גם הרבה תביעות וגם הרבה נהגים יקרים. ההשערה שלנו: טנדר הוא רכב עבודה שנוסעים בו הרבה, ולכן ייתכן שחלק מההשפעה שלו היא בעצם השפעה של סוג השימוש, נבדוק זאת בעמודה usage_type.

## עמודה O: vehicle_age
גיל הרכב בשנים. זו עמודה מספרית שידועה ביום החתימה ולכן אין כאן דליפת מידע.

נבדוק ערכים חסרים וערכים לא הגיוניים, את צורת ההתפלגות ואת הקשר לעלות התביעות.

In [4]:
# בדיקת ערכים חסרים וסטטיסטיקה תיאורית של גיל הרכב
print("Missing:", df["vehicle_age"].isna().sum())
df["vehicle_age"].describe().round(1)

Missing: 0


count    45000.0
mean         7.6
std          5.3
min          0.0
25%          4.0
50%          6.0
75%         10.0
max         25.0
Name: vehicle_age, dtype: float64

הממוצע גבוה מהחציון, נבדוק כמה רכבים יש בכל גיל והאם יש ערכים חריגים

In [5]:
df["vehicle_age"].value_counts().sort_index()

vehicle_age
0      723
1     2269
2     3489
3     4037
4     4176
5     4099
6     3862
7     3478
8     3017
9     2606
10    2241
11    1943
12    1652
13    1286
14    1158
15     875
16     786
17     606
18     498
19     414
20     312
21     268
22     249
23     185
24     163
25     608
Name: count, dtype: int64

In [6]:
# חלוקת הרכבים לחמש קבוצות שוות לפי גיל הרכב והשוואת עלות התביעות בין הקבוצות
age_group = pd.qcut(df["vehicle_age"], 5)
df.groupby(age_group, observed=True)["annual_claim_cost"].agg(
    avg_cost="mean",
    share_with_claim=lambda s: (s > 0).mean(),
    share_expensive=lambda s: (s > 10000).mean(),
    drivers="count",
).round(3)

,avg_cost,share_with_claim,share_expensive,drivers
vehicle_age,,,,
"(-0.001, 3.0]",4180.330,0.308,0.143,10518
"(3.0, 5.0]",3569.221,0.331,0.118,8275
"(5.0, 8.0]",3040.877,0.341,0.091,10357
"(8.0, 12.0]",2326.196,0.379,0.058,8442
"(12.0, 25.0]",1707.849,0.415,0.034,7408


אנו רואים שיש קפיצה גדולה בכמות הנהגים בין רכבים בני 24 ל25 שנים.
נבדוק האם לגיל 25 יש השפעה שונה מבחינת התדירות או החומרה של התביעות

In [8]:
# השוואת עלות התביעות בגילאי הרכב הגבוהים, כדי לבדוק אם גיל 25 שונה מהגילאים שלפניו
old_cars = df[df["vehicle_age"] >= 20]
old_cars.groupby("vehicle_age")["annual_claim_cost"].agg(
    avg_cost="mean",
    share_with_claim=lambda s: (s > 0).mean(),
    drivers="count",
).round(3)

,avg_cost,share_with_claim,drivers
vehicle_age,,,
20,1698.798,0.458,312
21,1855.563,0.381,268
22,1491.189,0.446,249
23,1381.708,0.427,185
24,1366.006,0.405,163
25,1767.804,0.479,608


### מסקנה לגבי vehicle_age
העמודה נקייה, עם ערכים שלמים בין 0 ל-25 וללא ערכים חסרים. ההתפלגות מוטה ימינה (ממוצע 7.6, חציון 6): רוב הרכבים צעירים, עם זנב של רכבים ישנים. גיל 0 הוא רכב חדש ואינו שגיאה.

בגיל 25 יש קפיצה חדה במספר הרכבים (608 לעומת 163 בגיל 24). העלות ואחוז התביעות בגיל 25 נמצאים בטווח הרגיל של גילאי 20 עד 24, ולכן כנראה 25 מייצג - ומעלה

ככל שהרכב ישן יותר, התביעות שכיחות יותר אך זולות יותר: אחוז התביעות עולה מ-30.8% ל-41.5%, אחוז הנהגים היקרים יורד מ-14.3% ל-3.4%, והעלות הממוצעת יורדת פי 2.4. העמודה תישאר. השערה: רכב חדש יקר לתיקון בגלל השווי שלו. נבדוק זאת בעמודה vehicle_value.

חוקים לפונקצית הניקוי: ערך שאינו מספר או שלילי הופך לחסר, וערך מעל 25 הופך ל-25.

## עמודה P: engine_hp
עוצמת המנוע בכוחות סוס. זו עמודה מספרית שידועה ביום החתימה ולכן אין כאן דליפת מידע.

נבדוק ערכים חסרים וערכים לא הגיוניים, האם העוצמה הגיונית לפי סוג הרכב, ואת הקשר לעלות התביעות.

In [10]:
# בדיקת ערכים חסרים וסטטיסטיקה תיאורית של עוצמת המנוע
print("Missing:", df["engine_hp"].isna().sum())
df["engine_hp"].describe().round(1)

Missing: 0


count    45000.0
mean       143.9
std         66.4
min         12.0
25%         94.0
50%        137.0
75%        178.0
max        650.0
Name: engine_hp, dtype: float64

In [11]:
# בדיקה האם עוצמת המנוע הגיונית לפי סוג הרכב
df.groupby("vehicle_type")["engine_hp"].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
vehicle_type,,,,,,,,
motorcycle,3072.0,83.0,46.0,12.0,48.0,72.0,108.0,224.0
pickup,2985.0,219.0,49.0,110.0,185.0,218.0,252.0,410.0
sedan,13916.0,143.0,29.0,76.0,123.0,143.0,162.0,265.0
small,12227.0,86.0,16.0,45.0,74.0,85.0,96.0,148.0
sports,1840.0,321.0,92.0,171.0,254.0,306.0,369.0,650.0
suv,10960.0,176.0,36.0,90.0,151.0,176.0,201.0,318.0


עד כה כל הערכים נראים הגיונים ותקינים
נבדוק את השפעת העמודה על החומרה ותדירות התביעות

In [12]:
# חלוקת הרכבים לחמש קבוצות שוות לפי עוצמת המנוע והשוואת עלות התביעות בין הקבוצות
hp_group = pd.qcut(df["engine_hp"], 5)
df.groupby(hp_group, observed=True)["annual_claim_cost"].agg(
    avg_cost="mean",
    share_with_claim=lambda s: (s > 0).mean(),
    share_expensive=lambda s: (s > 10000).mean(),
    drivers="count",
).round(3)

,avg_cost,share_with_claim,share_expensive,drivers
engine_hp,,,,
"(11.999, 87.0]",1259.407,0.333,0.020,9077
"(87.0, 119.0]",1712.592,0.337,0.036,9057
"(119.0, 153.0]",2638.834,0.335,0.082,9059
"(153.0, 189.0]",3528.376,0.355,0.118,8910
"(189.0, 650.0]",6182.114,0.396,0.209,8897


נבדוק האם יש השפעה דומה לסוג הרכב ולכוחות הסוס

In [13]:
# השוואת הקשר בין עוצמת המנוע לעלות בכל הדאטה לעומת בתוך כל סוג רכב בנפרד
print("All vehicles:", round(df["engine_hp"].corr(df["annual_claim_cost"]), 3))
for t in df["vehicle_type"].unique():
    d = df[df["vehicle_type"] == t]
    print(t, round(d["engine_hp"].corr(d["annual_claim_cost"]), 3))

All vehicles: 0.254
suv 0.119
sedan 0.116
small 0.091
pickup 0.152
sports 0.149
motorcycle 0.243


### מסקנה לגבי engine_hp
העמודה ללא ערכים חסרים, עם טווח של 12 עד 650 כוחות סוס. הטווח הגיוני בתוך כל סוג רכב: מקטנועים ועד אופנועים כבדים, ועד מכוניות-על בקרב רכבי הספורט. אין ערכים בלתי אפשריים, והסדר בין הסוגים הגיוני.

ככל שהמנוע חזק יותר העלות גבוהה יותר, פי 4.9 בין החמישית החלשה לחזקה. אחוז התביעות כמעט לא משתנה (33% עד 40%), אך אחוז הנהגים היקרים עולה פי 10 (2% עד 21%). כלומר, מנוע חזק לא גורם להרבה יותר תאונות אלא לתאונות יקרות יותר.

חלק מהקשר נובע מסוג הרכב (משתנה מבלבל): הקורלציה עם העלות היא 0.254 בכל הדאטה, ו-0.09 עד 0.24 בתוך כל סוג רכב בנפרד. הקשר נשאר חיובי בכל ששת הסוגים, ולכן לעמודה תישאר vehicle_type.

חוק לפונקצית הניקוי: ערך שאינו מספר או שאינו חיובי הופך לחסר.

## עמודה P: vehicle_value

In [14]:
# פונקציית הניקוי של עמודות הרכב והשימוש שנוסיף לה חוק אחרי כל עמודה שנבדוק
def clean_vehicle_cols(df):
    df = df.copy()
    
    # סוג רכב: הסרת רווחים ואותיות קטנות, וערך שאינו אחד מששת הסוגים החוקיים הופך לחסר
    valid_vehicle = ["small", "sedan", "suv", "sports", "pickup", "motorcycle"]
    df["vehicle_type"] = df["vehicle_type"].astype(str).str.strip().str.lower()
    df.loc[~df["vehicle_type"].isin(valid_vehicle), "vehicle_type"] = np.nan

    # גיל רכב: המרה למספר, ערך שלילי הופך לחסר, וערך מעל עשרים וחמש הופך לעשרים וחמש
    df["vehicle_age"] = pd.to_numeric(df["vehicle_age"], errors="coerce")
    df.loc[df["vehicle_age"] < 0, "vehicle_age"] = np.nan
    df["vehicle_age"] = df["vehicle_age"].clip(upper=25)

    # עוצמת מנוע: המרה למספר, וערך שאינו מספר או שאינו חיובי הופך לחסר
    df["engine_hp"] = pd.to_numeric(df["engine_hp"], errors="coerce")
    df.loc[df["engine_hp"] <= 0, "engine_hp"] = np.nan
    
    return df

# בדיקה שהפונקציה רצה על כל הדאטה ועל שורה בודדת
print(clean_vehicle_cols(df).shape)
print(clean_vehicle_cols(df.iloc[[0]]).shape)

(45000, 30)
(1, 30)
